In [ ]:
!pip install spektral

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from spektral.data import SingleLoader, BatchLoader, Dataset, Graph
from spektral.layers import GCNConv
from spektral.models.gcn import GCN 

from sklearn.metrics import confusion_matrix, roc_auc_score

import time


import scipy.sparse as sp
from spektral.utils import gcn_filter

import os
import shutil 


import gc
import json


from spektral.datasets.utils import DATASET_FOLDER

from scipy import sparse
from scipy.special import softmax

In [2]:
class GCNConv_preprocess_adjacencyMatrix(object):
    """
    Applies the `gcn_filter` function of a GCN Layer to the adjacency
    matrix. The result depends on symmetry of adjacency matrix.

    **Arguments**

    - `layer_class`: the class of a layer from `spektral.layers.convolutional`,
    in order to verify if it is a GCNConv.
    - `symmetric`: boolean, indicates if adjacency matrix is symmetric (undirected graph),
    or if it is non-symmetric (directed graph).
    """

    def __init__(self, layer_class, symmetric):
        self.layer_class = layer_class
        self.symmetric = symmetric

    def __call__(self, graph):
        if self.layer_class == GCNConv:
            if self.symmetric:
                graph.a = gcn_filter(graph.a)
            else:
                graph.a = gcn_filter(graph.a, symmetric=False)
            return graph
        else:
            raise ValueError('The parameter must be GCNConv. For other convolutional layers, find the appropriate preprocessing')


In [3]:
def _preprocess_features(features):
    """
    Copy from https://github.com/danielegrattarola/spektral/blob/39fe897c5c06ce8bd8100e10fe9d373b91958cc7/spektral/datasets/citation.py#L192
    """
    rowsum = np.array(features.sum(1))
    r_inv = np.power(rowsum, -1).flatten()
    r_inv[np.isinf(r_inv)] = 0.0
    r_mat_inv = sp.diags(r_inv)
    features = r_mat_inv.dot(features)
    return features


class GCNConv_preprocess_features(object):
    """
    Applies the `_preprocess_features` to the node features.
    
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def __call__(self, graph):
        graph.x = _preprocess_features(graph.x)
        return graph
       

In [4]:
def predicciones(loader, nombre):
    """
    Calcula las predicciones y métricas de interés
    """
    TP_list = []
    FN_list = []
    FP_list = []
    TN_list = []
    acc = []
    prec = []
    rec = []
    esp = []
    F1 = []
    auc = []
    
    for k in range(loader.steps_per_epoch):
        inputs,target = loader.__next__()
        y_prediction = model(inputs, training=False)
        y_prediction = np.argmax(np.vstack(y_prediction), axis = 1)
        y_true=np.argmax(np.vstack(target), axis=1)
        prediccion=pd.DataFrame({"true_label":y_true, "prediction":y_prediction})
        prediccion.to_csv(os.path.join(prediccionesDirectorio,f'prediccion_{str(nombre)}_{k:03d}.csv'), index = None)
        
        #Create confusion matrix and normalizes it over predicted (columns)
        result = tf.math.confusion_matrix(y_true, y_prediction, num_classes=NUM_COMMUNITIES) 

        # confusion_matrix = [[TP, FN],
        #                     [FP, TN]]
        TP = result[0,0].numpy()
        FN = result[0,1].numpy()
        FP = result[1,0].numpy()
        TN = result[1,1].numpy()

        accuracy = (TP+TN)/(TP+FP+FN+TN)
        precision = TP/(TP+FP)
        recall = TP/(TP+FN)
        specificity = TN/(TN+FP)
        f1 = (2*precision*recall)/(precision+recall)
        auc_score = roc_auc_score(y_true, y_prediction)
        
        TP_list.append(TP)
        FN_list.append(FN)
        FP_list.append(FP)
        TN_list.append(TN)
        acc.append(accuracy)
        prec.append(precision)
        rec.append(recall)
        esp.append(specificity)
        F1.append(f1)
        auc.append(auc_score)
        
    df = pd.DataFrame({"TP":TP_list, "FN":FN_list, "FP":FP_list, "TN":TN_list, "accuracy":acc, "precision":prec, "recall":rec, "specificity":esp, "f1":F1, "auc_score":auc})
    df.to_csv(os.path.join(metricasDirectorio, f'metricas_{str(nombre)}.csv'), index = None)



In [5]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True)
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    print(e)


1 Physical GPUs, 1 Logical GPUs


# cacic-springer (sinteticos)

In [ ]:
def save_parameters(parameters, filename):
    with open(filename, 'w') as f:
        json.dump(parameters, f)
        
def load_parameters(filename):
    with open(filename, 'r') as f:
        return json.load(f)


def generate_synthetic_graph_csbm(num_nodes, num_communities, num_features, edge_prob_matrix, node_features_mean,\
                                  feature_cov_matrix=None, directed=False, num_class1=None, seed=None):
    np.random.seed(seed)
    
    # Assign nodes to communities
    if num_class1 is None:
        communities = np.random.randint(0, num_communities, num_nodes) # caso balanceado
    else:
        indices = np.random.choice(num_nodes, num_class1, replace=False)
        communities = np.array([int(j in indices) for j in range(num_nodes)])
    
    # Generate node features
    if feature_cov_matrix is None:
        feature_cov_matrix = np.eye(num_features)
    features = np.zeros((num_nodes, num_features))
    for k in range(num_communities):
        nodes_in_community = np.where(communities == k)[0]
        features[nodes_in_community] = np.random.multivariate_normal(node_features_mean[k], feature_cov_matrix,\
                                                                     len(nodes_in_community))

    # Compute community membership probabilities based on node features
    community_membership_probs = softmax(features @ node_features_mean.T, axis=1)
    
    # Generate edges based on community membership probabilities
    adjacency_matrix = np.zeros((num_nodes, num_nodes))
    if directed:
        for i in range(num_nodes):
            for j in range(num_nodes):
                if i == j:
                    continue
                community_i = communities[i]
                community_j = communities[j]
                edge_prob = edge_prob_matrix[community_i, community_j] * community_membership_probs[i, community_j] * community_membership_probs[j, community_i]
                adjacency_matrix[i, j] = np.random.binomial(1, edge_prob)
    else: 
        for i in range(num_nodes):
            for j in range(i, num_nodes):
                if i == j:
                    continue
                community_i = communities[i]
                community_j = communities[j]
                edge_prob = edge_prob_matrix[community_i, community_j] * community_membership_probs[i, community_j] * community_membership_probs[j, community_i]
                adjacency_matrix[i, j] = adjacency_matrix[j, i] = np.random.binomial(1, edge_prob)

    labels = tf.keras.utils.to_categorical(communities)
    adjacency_matrix = sparse.csr_matrix(adjacency_matrix)
    return Graph(x=features, a=adjacency_matrix, y=labels)



class CacicSpringer_SyntheticGraphs(Dataset): # modificacion 2024 para cacic-springer
    
    def __init__(self, num_graphs, num_nodes, num_communities, num_features, edge_prob_matrix, \
                 node_features_mean, feature_cov_matrix=None, directed=False, class1_percent=None, \
                 seed=None, flattened=False, adjnul=False, **kwargs):
        self.num_graphs = num_graphs
        self.num_nodes = num_nodes
        self.num_communities = num_communities
        self.num_features = num_features
        self.edge_prob_matrix = edge_prob_matrix
        self.node_features_mean = node_features_mean
        self.feature_cov_matrix = feature_cov_matrix
        self.directed = directed
        self.class1_percent = class1_percent
        self.seed = seed
        self.flattened = flattened
        self.adjnul = adjnul
        super().__init__(**kwargs)

    @property
    def path(self):
        edge_prob_matrix_str = '-'.join(map(str, self.edge_prob_matrix.flatten()))            
        node_features_mean_str = '-'.join(map(str, self.node_features_mean.flatten()))
        
        if self.feature_cov_matrix is None:
            feature_cov_matrix_str = "None"
        else:
            feature_cov_matrix_str = '-'.join(map(str, self.feature_cov_matrix.flatten()))
            
        if self.directed:
            dir_string = "Directed"
        else:
            dir_string = "Undirected"
        
        if self.class1_percent is None:
            class1_string = "Balanced"
        else:
            class1_string = f'{self.class1_percent}PercentClass1'

        dirname = f'{self.num_graphs}Graphs_{self.num_nodes}Nodes_{self.num_communities}Classes_\
{self.num_features}Features_{edge_prob_matrix_str}EdgeProbMatrix_{node_features_mean_str}NodeFeaturesMean_\
{dir_string}_{class1_string}_{feature_cov_matrix_str}FeaturesCov'

        return os.path.join(DATASET_FOLDER, "CacicSpringer_SyntheticGraphs", dirname)
        
        
    @property
    def num_class1(self):
        if self.class1_percent is None:
            return None
        else:
            return int( (self.class1_percent * self.num_nodes) / 100 )
    
    
    # Getter para obtener los argumentos utilizados
    def get_args(self):
        return {"num_graphs": self.num_graphs, "num_nodes": self.num_nodes, \
                "num_communities": self.num_communities, "num_features": self.num_features, \
                "edge_prob_matrix": self.edge_prob_matrix, "node_features_mean": self.node_features_mean, \
                "feature_cov_matrix": self.feature_cov_matrix, "directed": self.directed, \
                "class1_percent": self.class1_percent, "seed": self.seed, \
                "flattened": self.flattened, "adjnul": self.adjnul}

    def download(self):
        os.makedirs(self.path)
        
        parameters = self.get_args()
        parameters["edge_prob_matrix"] = self.edge_prob_matrix.tolist()
        parameters["node_features_mean"] = self.node_features_mean.tolist()
        
        if self.seed is None:
            vector_seed = [None] * self.num_graphs
        else:
            assert len(self.seed) >= self.num_graphs, f'There are not enough seeds ({len(self.seed)}) for graphs ({self.num_graphs})'
            vector_seed = self.seed[:self.num_graphs] 
            parameters["seed"] = vector_seed
        
        graphs = [generate_synthetic_graph_csbm(self.num_nodes, self.num_communities, self.num_features, \
                                                self.edge_prob_matrix, self.node_features_mean, \
                                                self.feature_cov_matrix, self.directed, \
                                                self.num_class1, vector_seed[i]) \
                  for i in range(self.num_graphs)]
        
        for j in range(self.num_graphs):
            filename = os.path.join(self.path, f'graph_{j:03d}.npz')
            np.savez(filename, x=graphs[j].x, a=graphs[j].a, y=graphs[j].y)
        
        save_parameters(parameters, os.path.join(self.path,"parameters.json"))
        # Free memory
        del graphs
        gc.collect()

        

    def read(self): 
        if os.path.exists(self.path):
            parameters = load_parameters(os.path.join(self.path,"parameters.json"))
        
        #### TATI: if self.seed==None entonces debería generar nuevos (no leer lo que ya está de antes).
        ####       Lo mismo si cambia alguna semilla ####
        if ((self.seed is None) or 
            (self.seed is not None and parameters["seed"] != self.get_args()["seed"][:self.num_graphs])):
            self.delete()
            self.download()           

        # We must return a list of Graph objects
        output = []

        for j in range(self.num_graphs):
            data = np.load(os.path.join(self.path, f'graph_{j:03d}.npz'), allow_pickle=True)
            #### if flattened, entonces en lugar de leer las features guardadas, asigna una matriz de todos 1 ####
            if self.flattened:
                x_features = np.ones((self.num_nodes, self.num_features))
            else:
                x_features = data['x']
            #### if adjnul, entonces los nodos no estan conectados. Solo influirian las features disponibles ####
            if self.adjnul:
                matrix = np.zeros((self.num_nodes, self.num_nodes))
                adj_matrix = sparse.csr_matrix(matrix)
            else:
                adj_matrix = data['a'][()] # también puede ser a=data['a'].item()
            output.append(
                Graph(x=x_features, a=adj_matrix, y=data['y']) 
            )

        return output
    
    
    def delete(self):
        if os.path.exists(self.path):
            shutil.rmtree(self.path)


In [ ]:
def deco(string_list):
    edge = string_list[0]
    mean = string_list[1]
    if edge == "A":
        edge_prob_matrix = np.array([[0.9,0.1], [0.2,0.8]])
    elif edge == "B":
        edge_prob_matrix = np.array([[0.8,0.2], [0.3,0.7]])
    elif edge == "C":
        edge_prob_matrix = np.array([[0.6,0.4], [0.4,0.6]])
    elif edge == "D":
        edge_prob_matrix=np.array([[0.5,0.5], [0.5,0.5]])
    else:
        raise ValueError("no valid entry")
    if mean == "I":
        features_mean_matrix = np.array([[3,0], [0,3]])
    elif mean == "II":
        features_mean_matrix = np.array([[2,1], [1,2]])
    elif mean == "III":
        features_mean_matrix = np.array([[1.5,1], [1,1.5]])
    elif mean == "IV":
        features_mean_matrix = np.array([[1,1], [1,1]])
    else:
        raise ValueError("no valid entry")
    return edge_prob_matrix, features_mean_matrix



def instancia(clase, num_graphs, num_nodes, num_communities, num_features, edge_prob_matrix, \
                node_feature_means, feature_cov_matrix=None, directed=False, class1_percent=None, \
                seed=None, flattened=False, adjnul=False, symmetricAdjacency=False, \
                preprocAdjacency=True, preprocFeatures=True):
    """
    Función para instanciar la clase que define los diferentes conjuntos de grafos sintéticos
    """
    if preprocAdjacency and preprocFeatures:
        inst = clase(num_graphs, num_nodes, num_communities, num_features, edge_prob_matrix, \
                node_feature_means, feature_cov_matrix, directed, class1_percent, seed, flattened, adjnul, \
                     transforms=[GCNConv_preprocess_adjacencyMatrix(GCNConv, symmetric=symmetricAdjacency), \
                                 GCNConv_preprocess_features()])
    elif preprocAdjacency and ~preprocFeatures:
        inst = clase(num_graphs, num_nodes, num_communities, num_features, edge_prob_matrix, \
                node_feature_means, feature_cov_matrix, directed, class1_percent, seed, flattened, adjnul, \
                     transforms=[GCNConv_preprocess_adjacencyMatrix(GCNConv, symmetric=symmetricAdjacency)])
    elif ~preprocAdjacency and preprocFeatures:
        inst = clase(num_graphs, num_nodes, num_communities, num_features, edge_prob_matrix, \
                node_feature_means, feature_cov_matrix, directed, class1_percent, seed, flattened, adjnul, \
                     transforms=[GCNConv_preprocess_features()])
    else:
        inst = clase(num_graphs, num_nodes, num_communities, num_features, edge_prob_matrix, \
                node_feature_means, feature_cov_matrix, directed, class1_percent, seed, flattened, adjnul)
    return inst



In [ ]:
config_str = list(map(str, str("A-I").split('-')))
EDGE_PROB_MATRIX, NODE_FEATURES_MEAN = deco(config_str)
print("EDGE_PROB_MATRIX = ", EDGE_PROB_MATRIX)
print("NODE_FEATURES_MEAN = ", NODE_FEATURES_MEAN)


In [ ]:
PATH_RDOS = "/mnt/ctu13/prueba_2024/rdo_synthetic"
CLASE = eval("CacicSpringer_SyntheticGraphs")
NUM_GRAPHS = 100
NUM_NODES = 100
NUM_COMMUNITIES = 2
NUM_FEATURES = 2
config_str = list(map(str, str("A-I").split('-'))) 
FEATURE_COV_MATRIX = None
DIRECTED = True
CLASS1_PERCENT = None

seed_aux = list(map(int, str("123-234-345-456-567-678-789-321-654-987").split('-'))) 
# para generar 90 semillas diferentes para armar 100 grafos, a partir de las 10 semillas dadas:
SEED = seed_aux.copy()
for i in range(6):
    seed_aux = list(np.array(seed_aux)*10)
    for j in range(10):
        SEED.append(int(seed_aux[j]))

for i in range(3):
    seed_aux = list(np.array(seed_aux)+100)
    for j in range(10):
        SEED.append(int(seed_aux[j]))

FLATTENED = False
ADJNUL = False
SYMMETRIC_ADJACENCY = False
PREPROC_ADJACENCY = True
PREPROC_FEATURES = True

In [ ]:
# Se carga el conjunto de grafos
dataset = instancia(CLASE, NUM_GRAPHS, NUM_NODES, NUM_COMMUNITIES, NUM_FEATURES, EDGE_PROB_MATRIX, \
                    NODE_FEATURES_MEAN, FEATURE_COV_MATRIX, DIRECTED, CLASS1_PERCENT, SEED, \
                    FLATTENED, ADJNUL, SYMMETRIC_ADJACENCY, PREPROC_ADJACENCY, PREPROC_FEATURES)


# Se almacena el modelo para una sola de las corridas, elegida al azar
guardarModelo = np.random.randint(NUM_GRAPHS)
modeloDirectorio = os.path.join(PATH_RDOS,f'prueba_{guardarModelo:03d}/modelo')
os.makedirs(modeloDirectorio, exist_ok = True)


# ctu13

In [6]:
class CTU13balanced_durRate(Dataset):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def download(self):
        os.mkdir(self.path)
        
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]
        
        # para poder descargar los archivos ncol de cada captura y las features de cada nodo

        features_links = ["ktAgJS22kBXcMsT", 
                          "qfM8pjZgDi9EFfd",
                          "34PWCn9JmdDA9HE",
                          "YtcCmiNEyPiYB4Y",
                          "D2HdaEZYcFz5ryt",
                          "cTY262moFLGjtmn",
                          "TbfPFbSKWqYqR7n",
                          "s2GTjFz8rNxbs4z",
                          "9dZPAENNACreDEK",
                          "bwR2Zrky49JjtgA",
                          "CmYc9JyBsHwzaYD",
                          "TNSkGJcq2CPoFtM",
                          "XwZFrQYzMLNJxAY"
        ]
        
        
        dur_rate = ["nFprrYtrZsW8Fwj",
                    "WEzZ9CYcFHPkYWk",
                    "wJHqrAgwbQLrFz3",
                    "fY7KkRzr6Re9BWP",
                    "4CHHTiYWqXCs9Ln",
                    "4orMAcGx9xitKLa",
                    "Jb9Qm35tG2QagkC",
                    "ikjFiFaMLWr8scL",
                    "Y2aSjmCxKxQ6oWZ",
                    "YZFQpTTL7cxLNgY",
                    "Z34rnW6YXMSLZKm",
                    "qA4KZ25WC4BwZKn",
                    "6ZiXnr99jLxE2oC"
        ]

        
        ncol_links = ["B5EBDnAr4z55cc9",
                      "Pz4ba4jn3nCNgAp",
                      "EbkwSBHyAkHmdHE",
                      "ttyoxLc36s7ABCB",
                      "R3b9fe25x6ncoaT",
                      "wFZ72f9kL3XFki6",
                      "7EcYp9ACPqkQqDs",
                      "YcTZCARwKCY2jiB",
                      "3cc8mcTZaEC9LGM",
                      "NDgw4PwXAwQKgb2",
                      "wY38ypkj7QSJYib",
                      "dEZYJ84z53ozZZo",
                      "NKdZfBX6DG9nB8o"            
        ]
        
        for i in range(len(captures)):
            # x = nodes features (Dur, Rate)
            # a = adjacency matrix
            # y =labels
            
            # Read files with nodes features (csv file) and connections between nodes (ncol file)
            x_4label = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{features_links[i]}/download', sep=",", header=0)
            x_4label = x_4label.sort_values("node")
            
            x_tmp = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{dur_rate[i]}/download', sep=",", header=0)
    
            a_tmp = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{ncol_links[i]}/download', sep=" ", header=None, names=["source", "target", "weight"])
            
            
            # Create dictionaries that identify each node and label with an integer
            node_idx = {name: idx for idx, name in enumerate(sorted(x_4label["node"].unique()))}
            
            # Change node names and label for their corresponding integer
            a_tmp["source"] = a_tmp["source"].apply(lambda name: node_idx[name])
            a_tmp["target"] = a_tmp["target"].apply(lambda name: node_idx[name])
            
            # Node features: (Dur, Rate)
            x = x_tmp.sort_values("node")[x_tmp.columns.difference(["node"], sort=False)].to_numpy()       
            x = x.astype(np.float32)
            
            # Separate source, target and weight to create a sparce matrix
            a_source = a_tmp[["source"]].to_numpy().T
            a_source = np.reshape(a_source, a_source.shape[-1])
            a_target = a_tmp[["target"]].to_numpy().T
            a_target = np.reshape(a_target, a_target.shape[-1])
            a_weight = a_tmp[["weight"]].to_numpy().T
            a_weight = np.reshape(a_weight, a_weight.shape[-1])
            # Adjacency matrix:
            a = sparse.csr_matrix((a_weight, (a_source, a_target)), shape=(x.shape[0], x.shape[0]), dtype=np.float32)
            
            # Label (sintético):
            y = []
            for j in range(x_4label.shape[0]):
                if (x_4label.iloc[j,4] > 2):  
                    y.append(np.array([0., 1.])) # clase 1 = "infected"
                else:
                    y.append(np.array([1., 0.])) # clase 0 = "normal"
            y = np.array(y)
            y.astype(np.float32)
        
        
            # Save in format npz
            filename = os.path.join(self.path, f'graph_201108{captures[i]}_durRate.npz')
            np.savez(filename, x=x, a=a, y=y)

            # Free memory
            del x_4label, x_tmp, x, a_tmp, a_source, a_target, a_weight, a, y
            gc.collect()


    def read(self):
        # We must return a list of Graph objects
        output = []
        
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]

        for i in captures:
            data = np.load(os.path.join(self.path, f'graph_201108{i}_durRate.npz'), allow_pickle=True)
            output.append(
                Graph(x=data['x'], a=data['a'][()], y=data['y']) # también puede ser a=data['a'].item()
            )

        return output
    
    


In [7]:
PATH_RDOS = "/mnt/ctu13/pruebas_2024/rdo_ctu13"
NUM_GRAPHS = 13
symmetricAdjacency = False
dataset = CTU13balanced_durRate(transforms=[GCNConv_preprocess_adjacencyMatrix(GCNConv, symmetric=symmetricAdjacency), \
                                 GCNConv_preprocess_features()])

# Se almacena el modelo para una sola de las corridas, elegida al azar
guardarModelo = np.random.randint(NUM_GRAPHS)
modeloDirectorio = os.path.join(PATH_RDOS,f'prueba_{guardarModelo:03d}/modelo')
os.makedirs(modeloDirectorio, exist_ok = True)

<ipython-input-3-00f287d511f0>:6: RuntimeWarning: divide by zero encountered in power
  r_inv = np.power(rowsum, -1).flatten()


# iscx2012

In [ ]:
class iscx2012(Dataset):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def download(self):
        os.mkdir(self.path)
        
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]
        
        # para poder descargar los archivos ncol de cada captura y las features de cada nodo

        features_links = ["ktAgJS22kBXcMsT", 
                          "qfM8pjZgDi9EFfd",
                          "34PWCn9JmdDA9HE",
                          "YtcCmiNEyPiYB4Y",
                          "D2HdaEZYcFz5ryt",
                          "cTY262moFLGjtmn",
                          "TbfPFbSKWqYqR7n",
                          "s2GTjFz8rNxbs4z",
                          "9dZPAENNACreDEK",
                          "bwR2Zrky49JjtgA",
                          "CmYc9JyBsHwzaYD",
                          "TNSkGJcq2CPoFtM",
                          "XwZFrQYzMLNJxAY"
        ]
        
        
        dur_rate = ["nFprrYtrZsW8Fwj",
                    "WEzZ9CYcFHPkYWk",
                    "wJHqrAgwbQLrFz3",
                    "fY7KkRzr6Re9BWP",
                    "4CHHTiYWqXCs9Ln",
                    "4orMAcGx9xitKLa",
                    "Jb9Qm35tG2QagkC",
                    "ikjFiFaMLWr8scL",
                    "Y2aSjmCxKxQ6oWZ",
                    "YZFQpTTL7cxLNgY",
                    "Z34rnW6YXMSLZKm",
                    "qA4KZ25WC4BwZKn",
                    "6ZiXnr99jLxE2oC"
        ]

        
        ncol_links = ["B5EBDnAr4z55cc9",
                      "Pz4ba4jn3nCNgAp",
                      "EbkwSBHyAkHmdHE",
                      "ttyoxLc36s7ABCB",
                      "R3b9fe25x6ncoaT",
                      "wFZ72f9kL3XFki6",
                      "7EcYp9ACPqkQqDs",
                      "YcTZCARwKCY2jiB",
                      "3cc8mcTZaEC9LGM",
                      "NDgw4PwXAwQKgb2",
                      "wY38ypkj7QSJYib",
                      "dEZYJ84z53ozZZo",
                      "NKdZfBX6DG9nB8o"            
        ]
        
        for i in range(len(captures)):
            # x = nodes features (Dur, Rate)
            # a = adjacency matrix
            # y =labels
            
            # Read files with nodes features (csv file) and connections between nodes (ncol file)
            x_4label = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{features_links[i]}/download', sep=",", header=0)
            x_4label = x_4label.sort_values("node")
            
            x_tmp = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{dur_rate[i]}/download', sep=",", header=0)
    
            a_tmp = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{ncol_links[i]}/download', sep=" ", header=None, names=["source", "target", "weight"])
            
            
            # Create dictionaries that identify each node and label with an integer
            node_idx = {name: idx for idx, name in enumerate(sorted(x_4label["node"].unique()))}
            
            # Change node names and label for their corresponding integer
            a_tmp["source"] = a_tmp["source"].apply(lambda name: node_idx[name])
            a_tmp["target"] = a_tmp["target"].apply(lambda name: node_idx[name])
            
            # Node features: (Dur, Rate)
            x = x_tmp.sort_values("node")[x_tmp.columns.difference(["node"], sort=False)].to_numpy()       
            x = x.astype(np.float32)
            
            # Separate source, target and weight to create a sparce matrix
            a_source = a_tmp[["source"]].to_numpy().T
            a_source = np.reshape(a_source, a_source.shape[-1])
            a_target = a_tmp[["target"]].to_numpy().T
            a_target = np.reshape(a_target, a_target.shape[-1])
            a_weight = a_tmp[["weight"]].to_numpy().T
            a_weight = np.reshape(a_weight, a_weight.shape[-1])
            # Adjacency matrix:
            a = sparse.csr_matrix((a_weight, (a_source, a_target)), shape=(x.shape[0], x.shape[0]), dtype=np.float32)
            
            # Label (sintético):
            y = []
            for j in range(x_4label.shape[0]):
                if (x_4label.iloc[j,4] > 2):  
                    y.append(np.array([0., 1.])) # clase 1 = "infected"
                else:
                    y.append(np.array([1., 0.])) # clase 0 = "normal"
            y = np.array(y)
            y.astype(np.float32)
        
        
            # Save in format npz
            filename = os.path.join(self.path, f'graph_201108{captures[i]}_durRate.npz')
            np.savez(filename, x=x, a=a, y=y)

            # Free memory
            del x_4label, x_tmp, x, a_tmp, a_source, a_target, a_weight, a, y
            gc.collect()


    def read(self):
        # We must return a list of Graph objects
        output = []
        
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]

        for i in captures:
            data = np.load(os.path.join(self.path, f'graph_201108{i}_durRate.npz'), allow_pickle=True)
            output.append(
                Graph(x=data['x'], a=data['a'][()], y=data['y']) # también puede ser a=data['a'].item()
            )

        return output


# Entrenamiento

In [8]:
NUM_COMMUNITIES = 2

# k-fold, con k=NUM_GRAPHS
for i in range(NUM_GRAPHS):
    graficasDirectorio = os.path.join(PATH_RDOS,f'prueba_{i:03d}/graficas')
    prediccionesDirectorio = os.path.join(PATH_RDOS,f'prueba_{i:03d}/predicciones')
    metricasDirectorio = os.path.join(PATH_RDOS,f'prueba_{i:03d}/metricas')
    
    os.makedirs(graficasDirectorio, exist_ok = True)
    os.makedirs(prediccionesDirectorio, exist_ok = True)
    os.makedirs(metricasDirectorio, exist_ok = True)
    
    tf.keras.backend.clear_session() 

    indices = np.concatenate((np.arange(i), np.arange(i+1, NUM_GRAPHS)))
    graphs4train = dataset[indices]
    test_dataset = dataset[i:i+1]
    
    idxs = np.random.permutation(len(graphs4train))
    split_va = int(0.99 * len(graphs4train))        # para 100 grafos al multiplicar por 0.99 me aseguro dejar 1 para validacion (para 10 grafos multiplicar por 0.9)
    idx_tr, idx_va = np.split(idxs, [split_va])
    train_dataset = graphs4train[idx_tr]
    val_dataset = graphs4train[idx_va]
    
    batch_size = 1
    n_epochs = 200
    
    # Se crean data loaders
    train_loader = BatchLoader(train_dataset, batch_size=batch_size, epochs=n_epochs, shuffle=False, node_level=True)   
    val_loader = SingleLoader(val_dataset, epochs=n_epochs)
    test_loader = SingleLoader(test_dataset, epochs=n_epochs)

    n_classes = NUM_COMMUNITIES
    model = GCN(n_labels=n_classes, channels=16)

    # Compila el model
    model.compile(optimizer=Adam(learning_rate=0.01), loss="binary_crossentropy", metrics=["accuracy"])

    # Se define early stopping para prevenir overfitting   
    if i==guardarModelo:
        callbacks_list = [
            EarlyStopping(
                monitor="val_loss",
                patience=10,
                verbose=1
                ),
            ModelCheckpoint(
                filepath=modeloDirectorio,
                monitor="val_loss",
                save_best_only=True,
                )
        ]
    else:
        callbacks_list = [
            EarlyStopping(
                monitor="val_loss",
                patience=10,
                verbose=1
                )
        ]

    # Entrenamiento
    history = model.fit(
        train_loader.load(),
        steps_per_epoch=train_loader.steps_per_epoch,
        epochs=n_epochs,
        validation_data=val_loader.load(),
        validation_steps=val_loader.steps_per_epoch,
        callbacks=callbacks_list                            
    )
    
    ## GRAFICAR
    res = pd.DataFrame(history.history)
    res.reset_index(inplace=True)
    res.rename(columns={'index': 'epoch'}, inplace=True)
    res.to_csv(os.path.join(graficasDirectorio,f'epochsResults.csv'),index = None)          
    
    sns.set_theme(style="whitegrid")
    line1 = sns.lineplot(x="epoch", y='loss', data=res, label='Training Loss')
    line2 = sns.lineplot(x="epoch", y='val_loss', data=res, label='Test Loss')
    scatter1 = sns.scatterplot(x="epoch", y='loss', data=res, marker='o', color='skyblue')
    scatter2 = sns.scatterplot(x="epoch", y='val_loss', data=res, marker='o', color='orange')
    plt.ylabel("Loss Value")
    plt.legend()
    plt.savefig(os.path.join(graficasDirectorio,f'loss.eps'), format='eps', dpi=800)                             
    plt.clf()
    
    sns.set_theme(style="whitegrid")
    line1 = sns.lineplot(x="epoch", y="accuracy", data=res, label='Training Accuracy')
    line2 = sns.lineplot(x="epoch", y="val_accuracy", data=res, label='Test Accuracy')
    scatter1 = sns.scatterplot(x="epoch", y="accuracy", data=res, marker='o', color='skyblue')
    scatter2 = sns.scatterplot(x="epoch", y="val_accuracy", data=res, marker='o', color='orange')
    plt.ylabel("Accuracy Value")
    plt.legend()
    plt.savefig(os.path.join(graficasDirectorio,f'accuracy.eps'), format='eps', dpi=800)              
    plt.clf()
    
    # PREDICCION
    loaders = [test_loader, val_loader, train_loader]
    names = ["test", "val", "train"]
    for j in range(len(loaders)):
        predicciones(loaders[j], names[j])



Epoch 1/200
 1/11 [=>............................] - ETA: 1:41 - loss: 0.6732 - accuracy: 0.7260

ResourceExhaustedError: Graph execution error:

2 root error(s) found.
  (0) RESOURCE_EXHAUSTED:  MemoryError: Unable to allocate 127. GiB for an array with shape (184901, 184901) and data type float32
Traceback (most recent call last):

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/ops/script_ops.py", line 270, in __call__
    ret = func(*args)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/autograph/impl/api.py", line 642, in wrapper
    return func(*args, **kwargs)

  File "/usr/local/lib/python3.8/dist-packages/tensorflow/python/data/ops/dataset_ops.py", line 1030, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))

  File "/usr/local/lib/python3.8/dist-packages/keras/engine/data_adapter.py", line 831, in wrapped_generator
    for data in generator_fn():

  File "/usr/local/lib/python3.8/dist-packages/spektral/data/loaders.py", line 100, in __next__
    return self.collate(nxt)

  File "/usr/local/lib/python3.8/di

In [12]:
!ls /mnt/iscx2012/

iscx2012.11.143.csv
